# 07 — Full Model Matrix, 5 Seeds, Bootstrap CIs

Extends 06: adds MLP (StandardScaler + MLPClassifier pipeline, scaler fitted on source train only), runs the 2x2 dataset matrix for rf/lgbm/mlp across 5 seeds (split and model seeds vary jointly; medians refitted per seed), and adds uncertainty: across-seed mean/std/min/max per cell, plus percentile bootstrap CIs at seed 42 (B=1000 for count-based metrics; B=200 for AUPRC — average-precision is O(n log n) per resample and cannot be vectorized). MLP config for this study: hidden (128, 64), ReLU, Adam, early stopping. Results append incrementally to Drive with resume-skip on (seed, source, model, target).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json
import numpy as np
import pandas as pd

BASE   = '/content/drive/MyDrive/drift-conference'
CACHE  = f'{BASE}/data/nfv2/cache'
RESULT = f'{BASE}/results/nfv2'
os.makedirs(RESULT, exist_ok=True)

CFG = dict(
    seeds         = [42, 43, 44, 45, 46],
    test_size     = 0.30,
    rf_estimators = 300,
    ece_bins      = 15,
    boot_B        = 1000,
    boot_B_auprc  = 200,
    boot_seed     = 42,
    mlp_hidden    = (128, 64),
    mlp_max_iter  = 100,
)
MAIN_CSV = f'{RESULT}/nfv2_matrix_seeds.csv'
BOOT_CSV = f'{RESULT}/nfv2_matrix_bootstrap.csv'
AGG_CSV  = f'{RESULT}/nfv2_matrix_aggregate.csv'
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

In [ ]:
d18 = pd.read_parquet(f'{CACHE}/nf2018v2_prepared.parquet')
dun = pd.read_parquet(f'{CACHE}/nfunswv2_prepared.parquet')
DATASETS = {'nf2018': d18, 'nfunsw': dun}
FEATURES = [c for c in d18.columns if c not in ('Label', 'Attack')]
assert [c for c in dun.columns if c not in ('Label', 'Attack')] == FEATURES
print('features:', len(FEATURES),
      '| nf2018:', d18.shape, '| nfunsw:', dun.shape)

In [ ]:
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                              brier_score_loss, confusion_matrix)

def ece_score(y_true, p_pos, n_bins=15):
    conf = np.maximum(p_pos, 1 - p_pos)
    correct = ((p_pos >= 0.5).astype(int) == y_true).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece

def all_metrics(y_true, p_pos):
    y_true = np.asarray(y_true); p_pos = np.asarray(p_pos)
    pred = (p_pos >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    ap_att = average_precision_score(y_true, p_pos)
    ap_ben = average_precision_score(1 - y_true, 1 - p_pos)
    return dict(
        macro_f1    = f1_score(y_true, pred, average='macro'),
        weighted_f1 = f1_score(y_true, pred, average='weighted'),
        mcc         = matthews_corrcoef(y_true, pred),
        auprc_macro = (ap_att + ap_ben) / 2,
        fp_rate     = fp / (fp + tn) if (fp + tn) else np.nan,
        brier       = brier_score_loss(y_true, p_pos),
        ece         = ece_score(y_true, p_pos, CFG['ece_bins']),
    )

def bootstrap_cis(y_true, p_pos, B, B_ap, rng):
    """Percentile 95% CIs by resampling test rows from precomputed predictions."""
    y = np.asarray(y_true); p = np.asarray(p_pos); n = len(y)
    keys = ['macro_f1', 'mcc', 'fp_rate', 'brier', 'ece']
    samples = {k: np.empty(B) for k in keys}
    samples['auprc_macro'] = np.empty(B_ap)
    for b in range(B):
        ix = rng.integers(0, n, n)
        yb, pb = y[ix], p[ix]
        pred = (pb >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(yb, pred, labels=[0, 1]).ravel()
        samples['macro_f1'][b] = f1_score(yb, pred, average='macro')
        samples['mcc'][b]      = matthews_corrcoef(yb, pred)
        samples['fp_rate'][b]  = fp / (fp + tn) if (fp + tn) else np.nan
        samples['brier'][b]    = np.mean((pb - yb) ** 2)
        samples['ece'][b]      = ece_score(yb, pb, CFG['ece_bins'])
        if b < B_ap:
            ap_a = average_precision_score(yb, pb)
            ap_b = average_precision_score(1 - yb, 1 - pb)
            samples['auprc_macro'][b] = (ap_a + ap_b) / 2
    out = {}
    for k, arr in samples.items():
        out[f'{k}_lo'] = float(np.percentile(arr, 2.5))
        out[f'{k}_hi'] = float(np.percentile(arr, 97.5))
    return out

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import lightgbm as lgb

F32_SAFE = 1e37

def clean_X(df, medians=None):
    X = df[FEATURES].astype('float64')
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.mask(X.abs() > F32_SAFE, np.nan)
    if medians is None:
        medians = X.median()
    return X.fillna(medians), medians

def make_models(seed):
    return {
        'rf':   RandomForestClassifier(n_estimators=CFG['rf_estimators'],
                                       n_jobs=-1, random_state=seed),
        'lgbm': lgb.LGBMClassifier(n_estimators=CFG['rf_estimators'],
                                   random_state=seed, n_jobs=-1, verbosity=-1),
        'mlp':  Pipeline([
                    ('scaler', StandardScaler()),
                    ('clf', MLPClassifier(hidden_layer_sizes=CFG['mlp_hidden'],
                                          activation='relu', solver='adam',
                                          batch_size=1024, max_iter=CFG['mlp_max_iter'],
                                          early_stopping=True, validation_fraction=0.05,
                                          n_iter_no_change=5, random_state=seed)),
                ]),
    }

In [ ]:
# Main loop: incremental append to Drive, resume-skip on (seed, source, model, target).
done = set()
if os.path.exists(MAIN_CSV):
    prev = pd.read_csv(MAIN_CSV)
    done = set(map(tuple, prev[['seed', 'source', 'model', 'target']].values))
    print(f'resume: {len(done)} cells already recorded')

boot_store = {}  # (source, model, target) -> (y, p) at boot_seed

for seed in CFG['seeds']:
    splits = {}
    for name, d in DATASETS.items():
        tr, te = train_test_split(d, test_size=CFG['test_size'],
                                  stratify=d['Attack'], random_state=seed)
        splits[name] = dict(train=tr.reset_index(drop=True),
                            test=te.reset_index(drop=True))
    for src in DATASETS:
        need_fit = [m for m in ('rf', 'lgbm', 'mlp')
                    if not all((seed, src, m, t) in done for t in DATASETS)] \
                   or ([m for m in ('rf', 'lgbm', 'mlp')] if seed == CFG['boot_seed'] else [])
        if not need_fit:
            continue
        Xtr, med = clean_X(splits[src]['train'])
        ytr = splits[src]['train']['Label'].values
        for mname, model in make_models(seed).items():
            recorded = all((seed, src, mname, t) in done for t in DATASETS)
            if recorded and seed != CFG['boot_seed']:
                continue
            t0 = time.time()
            model.fit(Xtr, ytr)
            print(f'seed {seed} | fit {mname} on {src}: {time.time()-t0:.0f}s')
            for tgt in DATASETS:
                te = splits[tgt]['test']
                Xte, _ = clean_X(te, medians=med)
                p = model.predict_proba(Xte)[:, 1]
                if seed == CFG['boot_seed']:
                    boot_store[(src, mname, tgt)] = (te['Label'].values.copy(), p.copy())
                if (seed, src, mname, tgt) not in done:
                    m = all_metrics(te['Label'].values, p)
                    m.update(seed=seed, source=src, model=mname, target=tgt,
                             setting=('in_domain' if src == tgt else 'zero_shot'))
                    pd.DataFrame([m]).to_csv(MAIN_CSV, mode='a', index=False,
                                             header=not os.path.exists(MAIN_CSV))
                    done.add((seed, src, mname, tgt))
                    print(f"  -> {tgt}: MCC={m['mcc']:.3f} macroF1={m['macro_f1']:.3f} "
                          f"ECE={m['ece']:.3f}")
            del model
            gc.collect()

print('main loop complete:', len(done), 'rows in', MAIN_CSV)

In [ ]:
# Bootstrap CIs at boot_seed from stored predictions.
rng = np.random.default_rng(CFG['boot_seed'])
boot_rows = []
for (src, mname, tgt), (y, p) in sorted(boot_store.items()):
    t0 = time.time()
    ci = bootstrap_cis(y, p, CFG['boot_B'], CFG['boot_B_auprc'], rng)
    ci.update(source=src, model=mname, target=tgt, seed=CFG['boot_seed'],
              B=CFG['boot_B'], B_auprc=CFG['boot_B_auprc'])
    boot_rows.append(ci)
    print(f'{src}/{mname}->{tgt}: {time.time()-t0:.0f}s  '
          f"MCC 95% CI [{ci['mcc_lo']:.3f}, {ci['mcc_hi']:.3f}]")
pd.DataFrame(boot_rows).round(4).to_csv(BOOT_CSV, index=False)
print('saved', BOOT_CSV)

In [ ]:
# Aggregate across seeds: mean / std / min / max per cell.
df = pd.read_csv(MAIN_CSV).drop_duplicates(['seed', 'source', 'model', 'target'])
metrics = ['macro_f1', 'weighted_f1', 'mcc', 'auprc_macro', 'fp_rate', 'brier', 'ece']
agg = (df.groupby(['source', 'model', 'target', 'setting'])[metrics]
         .agg(['mean', 'std', 'min', 'max']).round(4))
agg.columns = ['_'.join(c) for c in agg.columns]
agg = agg.reset_index()
agg.to_csv(AGG_CSV, index=False)
print('seeds per cell:', df.groupby(['source', 'model', 'target']).size().unique())
display(df.pivot_table(index=['model', 'source'], columns='target',
                       values='mcc', aggfunc='mean').round(3))
agg[['source', 'model', 'target', 'setting', 'mcc_mean', 'mcc_std',
     'macro_f1_mean', 'macro_f1_std', 'ece_mean', 'ece_std']]

In [ ]:
# Commit: strip outputs, stage named files only, gated, with proof.
import subprocess, glob as _glob
os.chdir(BASE)
nb = 'notebooks/07_full_matrix_seeds_cis.ipynb'
!jupyter nbconvert --ClearOutputPreprocessor.enabled=True --inplace {nb}
targets = [nb, 'results/nfv2/nfv2_matrix_seeds.csv',
           'results/nfv2/nfv2_matrix_bootstrap.csv',
           'results/nfv2/nfv2_matrix_aggregate.csv']
for t in targets:
    assert os.path.exists(t), f'missing {t}'
    !git add {t}
staged = subprocess.run(['git', 'diff', '--cached', '--name-only'],
                        capture_output=True, text=True).stdout.split()
assert set(staged) <= set(targets), f'unexpected staged: {set(staged) - set(targets)}'
!git commit -m "07: full 3-model NF-v2 matrix, 5 seeds, bootstrap CIs (B=1000; AUPRC B=200)"
!git push
!git log --oneline -1
!git status --short